# Ordered Logistic Regression Results for Adoption Predictors Exploration with `mlcroissant`
This notebook demonstrates how to load and analyze the FAIR² Dataset—"Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya"—using the `mlcroissant` library.

### Dataset Source
The dataset is defined by a Croissant schema at the following URL:

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`

The dataset contains ordered logistic regression outputs related to rangeland management interventions across multiple Kenyan counties. We will explore the record sets, fields, process the data, and perform exploratory data analysis.

In [ ]:
# Install the mlcroissant library if it's not already installed
!pip install mlcroissant

## 1. Data Loading
We'll load the schema and dataset metadata, then print a summary.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np

# Define the dataset Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}\n")
print(f"Published: {getattr(metadata, 'datePublished', 'N/A')}")
print(f"License: {getattr(metadata, 'license', 'N/A')}")
print(f"Spatial coverage: {getattr(metadata, 'spatialCoverage', 'N/A')}")
if hasattr(metadata, 'keywords'):
    print(f"Keywords: {', '.join(metadata.keywords)}")

## 2. Data Overview
Let's check what record sets and fields are defined in the dataset.

We list all available record sets by their `@id`, and for each record set, its respective fields (also using their `@id`).

In [ ]:
# Collect all record sets in the schema
record_sets = []
if hasattr(metadata, 'recordSet') and metadata.recordSet:
    for rec in metadata.recordSet:
        rec_id = getattr(rec, '@id', None)
        rec_name = getattr(rec, 'name', '(No name)')
        print(f"Record Set: {rec_name} (@id: {rec_id})")
        record_sets.append(rec_id)
        if hasattr(rec, 'field') and rec.field:
            print("  Fields:")
            for field in rec.field:
                fid = getattr(field, '@id', None)
                fname = getattr(field, 'name', '(No name)')
                print(f"    - {fname} (@id: {fid})")
        print()
else:
    # If no recordSet root, try to find via records() API
    print("No explicit record sets found in metadata. Inferring available record sets...\n")
    print("Available record sets:")
    record_sets = dataset.record_sets()
    for rec_id in record_sets:
        print(f"  - {rec_id}")

## 3. Data Extraction
Now, let's load the data from each available record set (using each record set's `@id`) into pandas DataFrames for analysis.

You can use the printed record sets and field `@id`s above to select specific sets or fields as needed.

In [ ]:
# Use record sets discovered above. If none, record_sets will be inferred again.
if not record_sets:
    record_sets = dataset.record_sets()

dataframes = {}
for rec_id in record_sets:
    print(f"\nLoading records from record set @id: {rec_id}")
    records = list(dataset.records(record_set=rec_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[rec_id] = df
        print(f"  Loaded shape: {df.shape}")
        print(f"  Columns: {list(df.columns)}")
        print(df.head(2))
    else:
        print("  No records found for this set.")

# Select the first populated record set for the next steps
populated_record_set = None
for rid, df in dataframes.items():
    if not df.empty:
        populated_record_set = rid
        break

if populated_record_set is None:
    print("No record sets with data found.")
else:
    print(f"\nPrimary data set used: {populated_record_set}")
    print("Columns:", dataframes[populated_record_set].columns.tolist())
    display(dataframes[populated_record_set].head())

## 4. Exploratory Data Analysis (EDA)
Let's explore, filter, and process a numeric field. You can use the column (field) `@id`s shown above.

We'll demonstrate filtering rows where a numeric variable exceeds a threshold, normalizing it, and grouping by a categorical field if available.

In [ ]:
# Select a DataFrame and a numeric column by @id
df = None
if dataframes and populated_record_set:
    df = dataframes[populated_record_set]
else:
    print("No loaded data for EDA.")

# Infer a numeric field
numeric_field = None
group_field = None
if df is not None:
    numerics = df.select_dtypes(include=[np.number]).columns
    if len(numerics) > 0:
        numeric_field = numerics[0]
        print(f"Selected numeric field for analysis: {numeric_field}")
    else:
        # Try to parse columns for possible numeric fields
        for col in df.columns:
            try:
                if pd.api.types.is_numeric_dtype(df[col]):
                    numeric_field = col
                    break
                # Try to coerce
                pd.to_numeric(df[col])
                numeric_field = col
                df[col] = pd.to_numeric(df[col], errors='coerce')
                break
            except Exception:
                continue

    # Try to get a non-numeric field for grouping
    for col in df.columns:
        if col != numeric_field and df[col].dtype == object:
            group_field = col
            print(f"Selected group field: {group_field}")
            break

if numeric_field and numeric_field in df.columns:
    # Filter where numeric_field > threshold
    threshold = df[numeric_field].mean() if np.issubdtype(df[numeric_field].dtype, np.number) else 10
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records where {numeric_field} > {threshold}:")
    display(filtered_df.head())

    # Normalize
    norm_col = f"{numeric_field}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"\nNormalized column '{numeric_field}' as '{norm_col}':")
    display(filtered_df[[numeric_field, norm_col]].head())

    # Group
    if group_field and group_field in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"\nGrouped mean of '{numeric_field}' by '{group_field}':")
        display(grouped_df.head())
else:
    print("No numeric field found for analysis.")

## 5. Visualization
Visualize the distribution of the chosen numeric field, and if applicable, its grouping by the selected group field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if df is not None and numeric_field in df.columns:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field].dropna(), kde=True, bins=30)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Frequency")
    plt.show()

    # Boxplot by group if available
    if group_field and group_field in df.columns:
        plt.figure(figsize=(10,4))
        sns.boxplot(x=group_field, y=numeric_field, data=df)
        plt.title(f"{numeric_field} by {group_field}")
        plt.xticks(rotation=35)
        plt.show()

## 6. Conclusion
In this notebook, we demonstrated loading and exploring a Croissant-described dataset using `mlcroissant`. We listed available record sets and fields, extracted and analyzed one set using its `@id`, and performed basic EDA including filtering, normalization, grouping, and visualization.

Next steps could include: more detailed field-level analyses, cross-dataset joins using field `@id`s, or advanced modeling.